In [ ]:
class Company
{
    string Name { get; set; }
    List<Employee> Employees { get; set; }
    List<Order> Orders { get; set; }
    List<Customer> Customers { get; set; }

    public Company(string name)
    {
        Name = name;
        Employees = new List<Employee>();
        Orders = new List<Order>();
    }

    public void add(Employee employee)
    {
        Employees.Add(employee);
    }

    public void add(Order order)
    {
        order.order_status_changed += order_status_changed;
        Orders.Add(order);
    }

    public void add(Customer customer)
    {
        Customers.Add(customer);
    }

    public void give_work(Employee employee, Order order)
    {
        if (!Employees.Contains(employee))
        {
            Console.WriteLine($"Данного сотрудника нет в компании \"{Name}\"");
            return;
        }

        if (!Orders.Contains(order))
        {
            Console.WriteLine($"Данного заказа нет в компании \"{Name}\"");
            return;
        }

        order.set_employee(employee);
        employee.order_received(order);
        order.set_status("В работе");
    }

    public void order_status_changed(Order order, Employee employee, string status)
    {
        Console.WriteLine($"Статус заказа с OrderId={order.OrderId}, за который ответственен сотрудник {employee.get_full_name()}, изменён на '{status}'");
    }

    public void display_orders_by(string status)
    {
        foreach (Order order in Orders)
        {
            if (order.OrderStatus == status)
            {
                order.display_info();
            }
        }
    }

    public void display_orders_by(Customer customer)
    {
        foreach (Order order in Orders)
        {
            if (order.customer == customer)
            {
                order.display_info();
            }
        }
    }
    public void display_report()
    {
        foreach(Employee employee in Employees)
        {
            Console.WriteLine($"{employee.get_full_name()}: {employee.OrdersProcessed}");
        }
    }
}

class Employee
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string Position { get; set; }
    public int OrdersProcessed { get; private set; }
    public List<Order> orders { get; private set; }

    public Employee(string first_name, string last_name, string position)
    {
        FirstName = first_name;
        LastName = last_name;
        Position = position;
        OrdersProcessed = 0;
        orders = new List<Order>();
    }

    public void order_received(Order order)
    {
        orders.Add(order);
    }

    public string get_full_name()
    {
        return $"'{FirstName} {LastName}'";
    }

    public void order_status_changed(Order order)
    {
        Console.WriteLine($"Сотрудник {get_full_name()} уведомлён об изменении статуса его заказа");
        
        if (order.OrderStatus == "Завершён")
        {
            OrdersProcessed++;
        }
    }
}

class Customer
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string PhoneNumber { get; set; }

    public Customer(string first_name, string last_name, string phone_number)
    {
        FirstName = first_name;
        LastName = last_name;
        PhoneNumber = phone_number;
    }

    public string get_full_name()
    {
        return $"{FirstName} {LastName}";
    }
}

class Order
{
    static int TotalOrderCount = 0;

    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Employee AssignedEmployee { get; set; }
    public string OrderStatus { get; private set; }
    public DateTime CreatedAt { get; private set; }
    public Customer customer { get; private set; }

    public delegate void order_status_changed_handler(Order order, Employee employee, string status);
    public event order_status_changed_handler order_status_changed;
    public delegate void order_status_changed_message(Order order);
    public order_status_changed_message send_message;

    public Order(string decription, Customer customer)
    {
        OrderId = TotalOrderCount++;
        Description = Description;
        OrderStatus = "Создан";
        CreatedAt = DateTime.Now;
        this.customer = customer;
    }

    public void set_employee(Employee employee)
    {
        send_message = employee.order_status_changed;
        AssignedEmployee = employee;
    }

    public void set_status(string status)
    {
        OrderStatus = status;
        order_status_changed?.Invoke(this, AssignedEmployee, status);
        send_message(this);
    }

    public void display_info()
    {
        Console.WriteLine($"OrderId={OrderId};Description={Description};employee.get_full_name={AssignedEmployee?.get_full_name()};OrderStatus={OrderStatus};CreatedAt={CreatedAt};customer.get_full_name={customer.get_full_name()}");
    }
}

Company company = new Company("Компания .inc");

Employee employee1 = new Employee("Андрей", "Дубов", "Сантехник");
Employee employee2 = new Employee("Николай", "Реков", "Газовщик");
company.add(employee1);
company.add(employee2);

Customer customer = new Customer("Олег", "Новоязов", "+7855843958345435");

Order order1 = new Order("Починить батарею", customer);
Order order2 = new Order("Починить кран", customer);
Order order3 = new Order("Проверить газовый котёл", customer);
company.add(order1);
company.add(order2);
company.add(order3);
Console.WriteLine("Список созданных заказов");
company.display_orders_by("Создан");
Console.WriteLine("");
company.give_work(employee1, order1);
Console.WriteLine("Список созданных заказов после назначения одного в работу");
company.display_orders_by("Создан");
Console.WriteLine("");
Console.WriteLine("Список заказов, которые в работе. Снова");
company.display_orders_by("В работе");
Console.WriteLine("");
company.give_work(employee1, order2);
company.give_work(employee2, order3);
Console.WriteLine("Список заказов по заказчику");
company.display_orders_by(customer);
Console.WriteLine("");
Console.WriteLine("Изменение статуса одного заказа");
order1.set_status("Завершён");
Console.WriteLine("Список сотрудников -> количетво выполненных заказов");
company.display_report();

Список созданных заказов
OrderId=0;Description=;employee.get_full_name=;OrderStatus=Создан;CreatedAt=12/3/2025 12:04:26 AM;customer.get_full_name=Олег Новоязов
OrderId=1;Description=;employee.get_full_name=;OrderStatus=Создан;CreatedAt=12/3/2025 12:04:26 AM;customer.get_full_name=Олег Новоязов
OrderId=2;Description=;employee.get_full_name=;OrderStatus=Создан;CreatedAt=12/3/2025 12:04:26 AM;customer.get_full_name=Олег Новоязов

Статус заказа с OrderId=0, за который ответственен сотрудник 'Андрей Дубов', изменён на 'В работе'
Сотрудник 'Андрей Дубов' уведомлён об изменении статуса его заказа
Список созданных заказов после назначения одного в работу
OrderId=1;Description=;employee.get_full_name=;OrderStatus=Создан;CreatedAt=12/3/2025 12:04:26 AM;customer.get_full_name=Олег Новоязов
OrderId=2;Description=;employee.get_full_name=;OrderStatus=Создан;CreatedAt=12/3/2025 12:04:26 AM;customer.get_full_name=Олег Новоязов

Список заказов, которые в работе. Снова
OrderId=0;Description=;employee.ge